# BIMBA Baseline — ReVA Video Inference

**Status: Experimental / not validated end-to-end.**

This notebook records the Role 2 baseline reproduction and provides preflight checks. The previous model load succeeded, but generation failed in Qwen attention due to inconsistent query/key/value dtypes. No valid prediction is recorded. Do not report this as a completed inference result.


## 1. Expected model artifacts

- Base: `lmms-lab/LLaVA-Video-7B-Qwen2`
- Adapter: `mmiemon/BIMBA-LLaVA-Qwen2-7B`
- Repository: BIMBA `BIMBA-LLaVA-NeXT`
- Earlier test GPU: NVIDIA Tesla T4

The Colab runtime reset. Repository files, virtual environment, model cache, and downloaded video were lost. The setup must be recreated before running inference.


In [ ]:
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

for p in ['/content/BIMBA/BIMBA-LLaVA-NeXT',
          '/content/bimba-env/bin/python',
          '/content/reva_video_cache',
          '/root/.cache/huggingface/hub']:
    print(p, '->', Path(p).exists())


## 2. Previously verified setup

The previous session used Python 3.10.22, PyTorch 2.1.2+cu121, Transformers 4.40.0.dev0, bitsandbytes 0.43.3, Accelerate 0.29.3, PEFT 0.10.0, Decord 0.6.0, PyAV 17.1.0, and Mamba-SSM 2.2.2. The custom CUDA extensions `causal_conv1d_cuda` and `selective_scan_cuda` imported successfully.

Earlier loading used 4-bit NF4 quantization and skipped quantization for `mm_projector` and `compressor`. The projector and compressor were manually moved to CUDA after loading. These changes are documented in `reports/baseline_handover.md`; they have not been retested in this fresh runtime.


## 3. Previous sample

- Video: `DJI_0157_d4_01.mp4`
- Duration: approximately 6.01 seconds
- Sampling: 16 frames, approximately uniform across the clip
- Question: “At what point in the video does the road with moving vehicles first appear?”
- Choices: A 5.0s; B 4.2s; C 2.5s; D 3.8s

**Prediction: none.** The generation call failed before a valid answer was returned. The answer must not be guessed from the choices or frame timestamps.


## 4. Known inference blocker

After model loading, generation failed inside Qwen2 attention: the query tensor was float16 while key and value tensors were float32. A forward pre-hook that cast the `lm_head` input did not fix the earlier attention failure.

Next debugging step: inspect the dtype and device of the first Qwen2 layer's `q_proj`, `k_proj`, `v_proj`, and `o_proj`, and trace the actual query/key/value tensors at the attention boundary. Do not add further speculative hooks without identifying where the dtype diverges.


## 5. Completion criteria

The baseline is complete only after all of the following are recorded:

1. Base and adapter load successfully in a fresh runtime.
2. One ReVA validation video completes generation.
3. The predicted answer is recorded alongside the ground-truth label, if available.
4. Runtime and peak GPU memory are recorded.
5. The inference script is rerun successfully from a clean session.
